# Ensemble Voting Filtering: Voting + Probability + Z-Score + P-Value

Notebook ini membuat shortlist akhir dari hasil voting ensemble.

Kriteria utama:
1. `Vote = 3/3`
2. Ranking berdasarkan `Predicted Probability`
3. Tambahan statistik prioritas: `z_score_all` dan `p_value_all`

Z-score dan p-value dihitung terhadap **semua pasangan dalam dataset**, bukan hanya `Vote = 3/3`, supaya kandidat kuat dibandingkan dengan seluruh ruang prediksi.


In [1]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import xml.etree.ElementTree as ET
import re
import math
from xml.sax.saxutils import escape

import pandas as pd

BASE = next(p / "2-machine_learning_analysis" for p in [Path.cwd(), *Path.cwd().parents] if (p / "2-machine_learning_analysis" / "2-predict").is_dir())
VOTING_DIR = BASE / "cache" / "screening" / "voting"
OUT_DIR = VOTING_DIR / 'shortlist'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_N = 100

FILES = {
    'ab66': VOTING_DIR / 'c_a51_ensemble.xlsx',
    'ab51': VOTING_DIR / 'c_ab66_ensemble.xlsx',
}
FILES


{'ab66': PosixPath('/home/admin_/projek-rki/2-predict/r_voting/c_a51_ensemble.xlsx'),
 'ab51': PosixPath('/home/admin_/projek-rki/2-predict/r_voting/c_ab66_ensemble.xlsx')}

## Helper XLSX

Notebook ini tidak bergantung pada `openpyxl`. Pembaca dan penulis XLSX dibuat sederhana lewat XML internal Excel.


In [2]:
XLSX_NS = {'a': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
REL_NS = {'pr': 'http://schemas.openxmlformats.org/package/2006/relationships'}
OFFICE_REL = 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'


def _col_index(cell_ref):
    match = re.match(r'([A-Z]+)', cell_ref)
    if not match:
        return 0
    value = 0
    for ch in match.group(1):
        value = value * 26 + ord(ch) - ord('A') + 1
    return value - 1


def _excel_col_name(index):
    name = ''
    index += 1
    while index:
        index, rem = divmod(index - 1, 26)
        name = chr(ord('A') + rem) + name
    return name


def _shared_strings(zf):
    try:
        root = ET.fromstring(zf.read('xl/sharedStrings.xml'))
    except KeyError:
        return []
    return [''.join(t.text or '' for t in si.findall('.//a:t', XLSX_NS)) for si in root.findall('a:si', XLSX_NS)]


def _first_sheet_path(zf):
    wb = ET.fromstring(zf.read('xl/workbook.xml'))
    rels = ET.fromstring(zf.read('xl/_rels/workbook.xml.rels'))
    targets = {rel.attrib['Id']: rel.attrib['Target'] for rel in rels.findall('pr:Relationship', REL_NS)}
    sheet = wb.find('a:sheets/a:sheet', XLSX_NS)
    rid = sheet.attrib[f'{{{OFFICE_REL}}}id']
    target = targets[rid]
    return target if target.startswith('xl/') else 'xl/' + target


def _cell_value(cell, shared):
    if cell.attrib.get('t') == 'inlineStr':
        return ''.join(t.text or '' for t in cell.findall('.//a:t', XLSX_NS))
    v = cell.find('a:v', XLSX_NS)
    if v is None:
        return ''
    text = v.text or ''
    if cell.attrib.get('t') == 's' and text.isdigit():
        idx = int(text)
        return shared[idx] if idx < len(shared) else text
    return text


def read_xlsx_basic(path):
    rows = []
    with ZipFile(path) as zf:
        shared = _shared_strings(zf)
        root = ET.fromstring(zf.read(_first_sheet_path(zf)))
    for row_node in root.findall('.//a:sheetData/a:row', XLSX_NS):
        values = []
        for cell in row_node.findall('a:c', XLSX_NS):
            idx = _col_index(cell.attrib.get('r', 'A1'))
            while len(values) <= idx:
                values.append('')
            values[idx] = _cell_value(cell, shared)
        rows.append(values)
    if not rows:
        return pd.DataFrame()
    header = [str(x).strip() for x in rows[0]]
    last = max((i for i, h in enumerate(header) if h), default=-1)
    header = header[:last + 1]
    data = []
    for raw in rows[1:]:
        row = raw[:len(header)] + [''] * max(0, len(header) - len(raw))
        if any(str(v).strip() for v in row):
            data.append(row)
    return pd.DataFrame(data, columns=header)


def _xlsx_cell(value, row_num, col_idx):
    ref = f'{_excel_col_name(col_idx)}{row_num}'
    text = '' if value is None else str(value)
    if text.lower() == 'nan':
        text = ''
    if len(text) > 32767:
        text = text[:32767]
    return f'<c r="{ref}" t="inlineStr"><is><t>{escape(text)}</t></is></c>'


def write_xlsx_basic(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    columns = [str(c) for c in df.columns]
    rows_xml = []
    rows_xml.append('<row r="1">' + ''.join(_xlsx_cell(c, 1, i) for i, c in enumerate(columns)) + '</row>')
    for row_num, row in enumerate(df.itertuples(index=False, name=None), start=2):
        rows_xml.append('<row r="%d">%s</row>' % (row_num, ''.join(_xlsx_cell(v, row_num, i) for i, v in enumerate(row))))
    last_col = _excel_col_name(max(len(columns) - 1, 0))
    last_row = max(len(df) + 1, 1)
    sheet_xml = ('<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
                 '<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">'
                 f'<dimension ref="A1:{last_col}{last_row}"/><sheetData>' + ''.join(rows_xml) + '</sheetData></worksheet>')
    workbook_xml = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships"><sheets><sheet name="shortlist" sheetId="1" r:id="rId1"/></sheets></workbook>'
    rels_xml = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/></Relationships>'
    workbook_rels_xml = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet1.xml"/></Relationships>'
    content_types_xml = '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types"><Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/><Default Extension="xml" ContentType="application/xml"/><Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/><Override PartName="/xl/worksheets/sheet1.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/></Types>'
    with ZipFile(path, 'w', ZIP_DEFLATED) as zf:
        zf.writestr('[Content_Types].xml', content_types_xml)
        zf.writestr('_rels/.rels', rels_xml)
        zf.writestr('xl/workbook.xml', workbook_xml)
        zf.writestr('xl/_rels/workbook.xml.rels', workbook_rels_xml)
        zf.writestr('xl/worksheets/sheet1.xml', sheet_xml)


def normal_sf(z):
    return 0.5 * math.erfc(float(z) / math.sqrt(2.0))


## Load Data Ensemble

In [3]:
dfs = {}
for name, path in FILES.items():
    print(f'Loading {name}: {path}')
    df = read_xlsx_basic(path)
    df['Predicted Probability'] = pd.to_numeric(df['Predicted Probability'], errors='coerce')
    df['Ensemble Label'] = pd.to_numeric(df['Ensemble Label'], errors='coerce').fillna(0).astype(int)
    df['vote_pos'] = df['Vote'].astype(str).str.extract(r'^(\d+)').astype(int)
    dfs[name] = df
    print(name, df.shape)


Loading ab66: /home/admin_/projek-rki/2-predict/r_voting/c_a51_ensemble.xlsx
ab66 (162890, 11)
Loading ab51: /home/admin_/projek-rki/2-predict/r_voting/c_ab66_ensemble.xlsx
ab51 (741955, 11)


## Hitung Z-Score dan P-Value dari Semua Pasangan

In [4]:
ranked = {}
summary = []
for name, df in dfs.items():
    s = df.copy()
    mu = s['Predicted Probability'].mean()
    sigma = s['Predicted Probability'].std(ddof=0)
    if pd.isna(sigma) or sigma == 0:
        sigma = 1e-12
    s['z_score_all'] = (s['Predicted Probability'] - mu) / sigma
    s['p_value_all'] = s['z_score_all'].map(normal_sf)
    s3 = s[s['vote_pos'] == 3].copy()
    s3 = s3.sort_values(['Predicted Probability', 'z_score_all', 'p_value_all'], ascending=[False, False, True]).reset_index(drop=True)
    s3['rank_global'] = range(1, len(s3) + 1)
    ranked[name] = s3
    summary.append({
        'dataset': name,
        'total_pairs': len(s),
        'vote3_pairs': len(s3),
        'mean_prob_all': mu,
        'std_prob_all': sigma,
        'vote3_max_prob': s3['Predicted Probability'].max(),
        'vote3_max_z_all': s3['z_score_all'].max(),
        'vote3_min_p_all': s3['p_value_all'].min(),
    })
pd.DataFrame(summary)


,dataset,total_pairs,vote3_pairs,mean_prob_all,std_prob_all,vote3_max_prob,vote3_max_z_all,vote3_min_p_all
0,ab66,162890,137057,0.892122,0.250685,0.99876,0.425385,0.335278
1,ab51,741955,609814,0.880895,0.259597,0.99875,0.453992,0.324917


## Ringkasan Voting dan Distribusi

In [5]:
vote_summary = []
for name, df in dfs.items():
    counts = df['Vote'].value_counts().to_dict()
    vote_summary.append({
        'dataset': name,
        'total': len(df),
        '3/3': counts.get('3/3', 0),
        '2/3': counts.get('2/3', 0),
        '1/3': counts.get('1/3', 0),
        '0/3': counts.get('0/3', 0),
        'ensemble_positive': int((df['Ensemble Label'] == 1).sum()),
    })
pd.DataFrame(vote_summary)


,dataset,total,3/3,2/3,1/3,0/3,ensemble_positive
0,ab66,162890,137057,8446,5755,11632,145503
1,ab51,741955,609814,45410,29605,57126,655224


In [6]:
prob_summary = []
for name, df in dfs.items():
    for vote, g in df.groupby('Vote'):
        prob_summary.append({
            'dataset': name,
            'vote': vote,
            'n': len(g),
            'min': g['Predicted Probability'].min(),
            'median': g['Predicted Probability'].median(),
            'p90': g['Predicted Probability'].quantile(0.90),
            'p95': g['Predicted Probability'].quantile(0.95),
            'p99': g['Predicted Probability'].quantile(0.99),
            'max': g['Predicted Probability'].max(),
        })
pd.DataFrame(prob_summary).sort_values(['dataset', 'vote'])


,dataset,vote,n,min,median,p90,p95,p99,max
4,ab51,0/3,57126,0.052582,0.110816,0.224002,0.259288,0.324749,0.461981
5,ab51,1/3,29605,0.196976,0.383501,0.503008,0.532149,0.595337,0.659720
6,ab51,2/3,45410,0.360860,0.685116,0.779690,0.800849,0.821301,0.831996
7,ab51,3/3,609814,0.530182,0.998319,0.998662,0.998687,0.998717,0.998750
0,ab66,0/3,11632,0.053140,0.111804,0.226479,0.261026,0.338709,0.434777
1,ab66,1/3,5755,0.201699,0.387063,0.502611,0.531307,0.595133,0.637826
2,ab66,2/3,8446,0.371845,0.681918,0.778584,0.801149,0.820575,0.829839
3,ab66,3/3,137057,0.537786,0.998456,0.998687,0.998708,0.998733,0.998760


## Buat 3 Shortlist per Dataset

Definisi:

- `top100`: 100 pasangan terbaik global dari `Vote = 3/3`.
- `top100peptide`: 100 pasangan terbaik dengan `PepID` unik.
- `top100protein`: 100 pasangan terbaik dengan `Protein (Uniprot_ID)` unik.

Semua diurutkan berdasarkan:
1. `Predicted Probability` tertinggi
2. `z_score_all` tertinggi
3. `p_value_all` terendah


In [7]:
KEEP_COLS = [
    'No',
    'PepID',
    'Seq peptida',
    'Snake Venom Protein',
    'Protein (Uniprot_ID)',
    'Cancer Protein',
    'Predicted Probability',
    'Vote',
    'Consensus',
    'Ensemble Label',
    'z_score_all',
    'p_value_all',
    'rank_global',
]

shortlists = {}
for name, df in ranked.items():
    top100 = df.head(TOP_N).copy()
    top100pep = df.drop_duplicates(subset=['PepID'], keep='first').head(TOP_N).copy()
    top100prot = df.drop_duplicates(subset=['Protein (Uniprot_ID)'], keep='first').head(TOP_N).copy()
    for d in (top100, top100pep, top100prot):
        d['No'] = range(1, len(d) + 1)
    shortlists[name] = {
        'top100': top100[KEEP_COLS],
        'top100peptide': top100pep[KEEP_COLS],
        'top100protein': top100prot[KEEP_COLS],
    }
    print(name)
    print('top100', top100.shape, 'top100peptide', top100pep.shape, 'top100protein', top100prot.shape)
    display(top100[KEEP_COLS])


ab66
top100 (100, 14) top100peptide (100, 14) top100protein (100, 14)


,No,PepID,Seq peptida,Snake Venom Protein,Protein (Uniprot_ID),Cancer Protein,Predicted Probability,Vote,Consensus,Ensemble Label,z_score_all,p_value_all,rank_global
0,1,P81115_pep6,KESEFR,Snaclec alboaggregin-B subunit alpha,Q7L576,Cytoplasmic FMR1-interacting protein 1,0.998760,3/3,Strong,1,0.425385,0.335278,1
1,2,P0C6B6_pep6,ERDLLK,Zinc metalloproteinase homolog-disintegrin alb...,Q9NTI5,Sister chromatid cohesion protein PDS5 homolog B,0.998755,3/3,Strong,1,0.425365,0.335285,2
2,3,P81115_pep6,KESEFR,Snaclec alboaggregin-B subunit alpha,Q96EP0,E3 ubiquitin-protein ligase RNF31,0.998754,3/3,Strong,1,0.425361,0.335287,3
3,4,P0C6B6_pep6,ERDLLK,Zinc metalloproteinase homolog-disintegrin alb...,Q96EP0,E3 ubiquitin-protein ligase RNF31,0.998753,3/3,Strong,1,0.425357,0.335288,4
4,5,P81115_pep3,TWEDAEK,Snaclec alboaggregin-B subunit alpha,Q7L576,Cytoplasmic FMR1-interacting protein 1,0.998753,3/3,Strong,1,0.425357,0.335288,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,P0CJ41_pep12,DIMLIK,Alpha-fibrinogenase albofibrase,P08581,Hepatocyte growth factor receptor,0.998746,3/3,Strong,1,0.425330,0.335298,96
96,97,P0DJF5_pep9,DIMLIK,Venom plasminogen activator GPV-PA,P08581,Hepatocyte growth factor receptor,0.998746,3/3,Strong,1,0.425330,0.335298,97
97,98,P0C6B6_pep6,ERDLLK,Zinc metalloproteinase homolog-disintegrin alb...,O75116,Rho-associated protein kinase 2,0.998746,3/3,Strong,1,0.425330,0.335298,98
98,99,P81115_pep6,KESEFR,Snaclec alboaggregin-B subunit alpha,O75116,Rho-associated protein kinase 2,0.998746,3/3,Strong,1,0.425330,0.335298,99


ab51
top100 (100, 14) top100peptide (100, 14) top100protein (100, 14)


,No,PepID,Seq peptida,Snake Venom Protein,Protein (Uniprot_ID),Cancer Protein,Predicted Probability,Vote,Consensus,Ensemble Label,z_score_all,p_value_all,rank_global
0,1,KAG8141327_pep9,VTLDLFGEWR,Putative snake venom metalloprotein partial,Q7L576,Cytoplasmic FMR1-interacting protein 1,0.998750,3/3,Strong,1,0.453992,0.324917,1
1,2,A0A2D0TC04_pep31,VMEVLK,Venom phosphodiesterase,Q7L576,Cytoplasmic FMR1-interacting protein 1,0.998750,3/3,Strong,1,0.453992,0.324917,2
2,3,A0A2D0TC04_pep31,VMEVLK,Venom phosphodiesterase,Q96EP0,E3 ubiquitin-protein ligase RNF31,0.998750,3/3,Strong,1,0.453992,0.324917,3
3,4,A0A2D0TC04_pep31,VMEVLK,Venom phosphodiesterase,P18206,Vinculin,0.998750,3/3,Strong,1,0.453992,0.324917,4
4,5,A0A2D0TC04_pep31,VMEVLK,Venom phosphodiesterase,Q9NTI5,Sister chromatid cohesion protein PDS5 homolog B,0.998749,3/3,Strong,1,0.453988,0.324919,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,AAB18384_pep3,LVPLFYK,Cardiotoxin-3b,Q9NTI5,Sister chromatid cohesion protein PDS5 homolog B,0.998742,3/3,Strong,1,0.453961,0.324928,96
96,97,AAC27690_pep2,LVPLFYK,Cardiotoxin-4b,Q9NTI5,Sister chromatid cohesion protein PDS5 homolog B,0.998742,3/3,Strong,1,0.453961,0.324928,97
97,98,P07525_pep3,LVPLFYK,1007132A cytotoxin D1,Q9NTI5,Sister chromatid cohesion protein PDS5 homolog B,0.998742,3/3,Strong,1,0.453961,0.324928,98
98,99,P60311_pep3,LVPLFYK,Cytotoxin KJC3,Q9NTI5,Sister chromatid cohesion protein PDS5 homolog B,0.998742,3/3,Strong,1,0.453961,0.324928,99


## Simpan XLSX

In [8]:
for name, items in shortlists.items():
    out_subdir = OUT_DIR / name
    out_subdir.mkdir(parents=True, exist_ok=True)
    for suffix, df in items.items():
        out = out_subdir / f'{name}_{suffix}.xlsx'
        write_xlsx_basic(df, out)
        print(out, len(df))


/home/admin_/projek-rki/2-predict/r_voting/shortlist/ab66/ab66_top100.xlsx 100
/home/admin_/projek-rki/2-predict/r_voting/shortlist/ab66/ab66_top100peptide.xlsx 100
/home/admin_/projek-rki/2-predict/r_voting/shortlist/ab66/ab66_top100protein.xlsx 100
/home/admin_/projek-rki/2-predict/r_voting/shortlist/ab51/ab51_top100.xlsx 100
/home/admin_/projek-rki/2-predict/r_voting/shortlist/ab51/ab51_top100peptide.xlsx 100
/home/admin_/projek-rki/2-predict/r_voting/shortlist/ab51/ab51_top100protein.xlsx 100


## Output Akhir

Struktur output:

```text
r_voting/shortlist/ab66/ab66_top100.xlsx
r_voting/shortlist/ab66/ab66_top100peptide.xlsx
r_voting/shortlist/ab66/ab66_top100protein.xlsx

r_voting/shortlist/ab51/ab51_top100.xlsx
r_voting/shortlist/ab51/ab51_top100peptide.xlsx
r_voting/shortlist/ab51/ab51_top100protein.xlsx
```
